# Lyric Engine - Kaggle Generation Workflow

This notebook handles cloning your codebase into Kaggle's `/kaggle/working` directory, and loading your PEFT model directly from an attached Kaggle Dataset in `/kaggle/input`.

## 1. One-Time Setup
Clone the repository to the writeable directory and ensure all pip packages are installed.

In [ ]:
import os

os.chdir('/kaggle/working')

if not os.path.exists('lyric-engine'):
    !git clone https://github.com/SMXFREEZE/lyric-engine
else:
    !git -C lyric-engine pull origin main

os.chdir('lyric-engine')

!pip install -q transformers peft accelerate bitsandbytes trl datasets \
    lyricsgenius pronouncing sentence-transformers textblob \
    fastapi uvicorn rich typer

print("Setup Complete.")

## 2. Configuration & Checkpoint Prep
Validates that the specific Kaggle Dataset path actually contains your `adapter_config.json` file.

In [ ]:
import os, torch

# ── HF Token ──────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    HF_TOKEN = _s.get_secret('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print("HF_TOKEN loaded ✓")
except Exception:
    HF_TOKEN = ''
    print("No HF_TOKEN secret (fine for public models)")

# ── Auto-discover checkpoint ──────────────────────────────────────────────────
INPUT_ROOT = '/kaggle/input'

def find_adapter(root: str) -> str | None:
    """Walk /kaggle/input up to 3 levels deep looking for adapter_config.json.
    Covers both /kaggle/input/<slug>/ and
    /kaggle/input/datasets/<user>/<slug>/ layouts.
    """
    for l1 in os.listdir(root):
        p1 = os.path.join(root, l1)
        if not os.path.isdir(p1): continue
        if os.path.exists(os.path.join(p1, 'adapter_config.json')): return p1
        for l2 in os.listdir(p1):
            p2 = os.path.join(p1, l2)
            if not os.path.isdir(p2): continue
            if os.path.exists(os.path.join(p2, 'adapter_config.json')): return p2
            for l3 in os.listdir(p2):
                p3 = os.path.join(p2, l3)
                if os.path.isdir(p3) and os.path.exists(os.path.join(p3, 'adapter_config.json')):
                    return p3
    return None

print(f"\nContents of {INPUT_ROOT}:")
for d in os.listdir(INPUT_ROOT):
    print(f"  /kaggle/input/{d}/  →  {os.listdir(os.path.join(INPUT_ROOT, d))[:6]}")

CKPT_DIR = find_adapter(INPUT_ROOT)

if CKPT_DIR is None:
    print("\n" + "="*60)
    print("CHECKPOINT NOT FOUND")
    print("="*60)
    print("Your dataset is not attached to this notebook yet.")
    print("\nTo fix:")
    print("  1. Click the folder icon (Input) in the right sidebar")
    print("  2. Click '+ Add Input' → search 'lyric-engine-trapcheckpoint'")
    print("  3. Click Add, then re-run this cell")
    raise FileNotFoundError("Attach the checkpoint dataset and re-run")

print(f"\nCheckpoint found: {CKPT_DIR} ✓")
print(f"  Files: {os.listdir(CKPT_DIR)}")

# ── Other config ──────────────────────────────────────────────────────────────
BASE_MODEL = 'mistralai/Mistral-7B-v0.1'
GENRE      = 'trap'
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory / 1e9
    for i in range(torch.cuda.device_count())
)
print(f"\nDevice: {device.upper()}  |  VRAM: {total_vram:.1f} GB  |  Model: {BASE_MODEL}")

## 3. Load Base Model and LoRA Adapter
Attaches your custom weights over the base 4-bit compressed Mistral model using the latest Transformers API.

In [ ]:
import os, sys, gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from safetensors.torch import load_file as _load_sf

sys.path.insert(0, '.')
from configs.genres import SPECIAL_TOKENS

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU {i}: {used:.1f} / {total:.1f} GB in use BEFORE load')

# ── 4-bit NF4 config ─────────────────────────────────────────────────────────
print("\nConfiguring 4-bit NF4 quantisation...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"\nLoading {BASE_MODEL} in 4-bit (low_cpu_mem_usage=True)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    token=HF_TOKEN or None,
)
print("Base model loaded ✓")

# ── Tokenizer ─────────────────────────────────────────────────────────────────
print("\nLoading tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
    print("  Found tokenizer in checkpoint ✓")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN or None)
    print("  Using base model tokenizer ✓")

tokenizer.pad_token = tokenizer.eos_token

new_tokens = [t for t in SPECIAL_TOKENS if t not in tokenizer.get_vocab()]
if new_tokens:
    print(f"  Adding {len(new_tokens)} Lyric Engine special tokens...")
    tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})

# ── Resize embeddings to exactly match checkpoint ─────────────────────────────
# The checkpoint stores embed_tokens / lm_head at whatever vocab size it was
# trained with. Read that size directly from the saved weights so we don't rely
# on having the identical SPECIAL_TOKENS list at inference time.
_sf_path = os.path.join(CKPT_DIR, 'adapter_model.safetensors')
_sf = _load_sf(_sf_path, device='cpu')
_embed_key = next(
    (k for k in _sf if 'embed_tokens.weight' in k or 'embedding.weight' in k),
    None
)
if _embed_key:
    ckpt_vocab_size = _sf[_embed_key].shape[0]
    del _sf
    if len(tokenizer) != ckpt_vocab_size:
        print(f"  Vocab mismatch: tokenizer={len(tokenizer)}, checkpoint={ckpt_vocab_size}")
        print(f"  Resizing embeddings → {ckpt_vocab_size}")
        base_model.resize_token_embeddings(ckpt_vocab_size)
    else:
        base_model.resize_token_embeddings(len(tokenizer))
else:
    del _sf
    base_model.resize_token_embeddings(len(tokenizer))

print(f"  Final vocab size: {base_model.config.vocab_size}")

# ── Apply LoRA adapter ────────────────────────────────────────────────────────
print(f"\nApplying LoRA adapter from:\n  {CKPT_DIR}")
trained_model = PeftModel.from_pretrained(base_model, CKPT_DIR)
trained_model.eval()
print("LoRA attached ✓")

for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU {i}: {used:.1f} / {total:.1f} GB in use AFTER load')

## 4. Full Lyric Generation Pipeline
Outputs lyrics with the loaded custom memory module.

In [ ]:
from src.inference.engine import LyricsEngine, SongMemory
import torch

engine = LyricsEngine(trained_model, tokenizer, device="cuda", beam_size=4)
memory = SongMemory(genre=GENRE, rhyme_scheme='AABB', target_syllables=10)
memory.sections.append(('[BUILD]', 'VERSE'))

print(f"\n======================================")
print(f"=== Generating full {GENRE.upper()} verse ===")
print(f"======================================\n")

with torch.no_grad():
    verse = engine.generate_verse(memory, num_lines=8, section='VERSE', arc_token='[BUILD]')

for line in verse:
    print(f"  {line}")